# 🌍 Émissions de CO₂ mondiales (1960–2023)
## Tendances, disparités et corrélations

**Auteur :** Diogo Almeida
**Date :** Septembre 2026
**Source :** [Our World in Data](https://github.com/owid/co2-data)

---

### Objectifs

1. Analyser les **tendances d'émission** sur 60 ans par grands blocs géographiques
2. Détecter les **ruptures** historiques (chocs pétroliers, Kyoto, COVID-19)
3. Étudier les **corrélations** PIB × émissions et le découplage
4. Comparer les **trajectoires** par pays et par source d'énergie
5. Exporter un fichier prêt pour le **dashboard Looker Studio**

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

DATA_RAW = Path("..") / "data" / "raw"
DATA_PROCESSED = Path("..") / "data" / "processed"
ASSETS = Path("..") / "assets"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

print(f"📁 Données dans : {DATA_RAW.resolve()}")

## 2. Chargement des données

Le dataset Our World in Data consolide des données de multiples sources :
- **Global Carbon Project** pour les émissions CO₂
- **Banque Mondiale** pour le PIB et la population
- **BP Statistical Review** pour les données énergétiques

In [ ]:
df = pd.read_csv(DATA_RAW / "owid-co2-data.csv")

print(f"Shape : {df.shape}")
print(f"Période : {df['year'].min()} – {df['year'].max()}")
print(f"Pays/entités : {df['country'].nunique()}")
display(df.head())

## 3. Nettoyage & préparation

Le dataset mélange des **pays** et des **agrégats** (World, Asia, EU-27). On les sépare.

In [ ]:
# Séparer pays et agrégats
aggregates = df[df["iso_code"].isna()]["country"].unique()
print(f"Agrégats : {len(aggregates)}")

df_countries = df[df["iso_code"].notna() & (df["year"] >= 1960)].copy()
df_world = df[(df["country"] == "World") & (df["year"] >= 1960)].copy()
df_regions = df[df["iso_code"].isna()].copy()

print(f"Pays : {df_countries['country'].nunique()} ({len(df_countries):,} lignes)")

In [ ]:
# Valeurs manquantes — colonnes clés
key_cols = ["co2", "co2_per_capita", "co2_per_gdp", "share_global_co2",
            "population", "gdp", "coal_co2", "oil_co2", "gas_co2"]
existing = [c for c in key_cols if c in df_countries.columns]

missing = (
    df_countries[existing].isnull().sum()
    .to_frame("missing")
    .assign(pct=lambda x: round(100 * x["missing"] / len(df_countries), 1))
    .sort_values("pct", ascending=False)
)
display(missing)

## 4. Analyse exploratoire

### 4.1 Émissions mondiales — Tendance globale

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_world["year"], y=df_world["co2"],
    mode="lines+markers", name="CO₂ total",
    line=dict(color="#D64045", width=2), marker=dict(size=3),
))
fig.update_layout(
    title="Émissions mondiales de CO₂ (1960–2023)",
    xaxis_title="Année", yaxis_title="CO₂ (Mt)",
    template="plotly_white", height=500,
)
events = {1973: "1er choc pétrolier", 1979: "2e choc pétrolier", 1991: "Chute URSS",
          1997: "Kyoto", 2008: "Crise financière", 2015: "Accord de Paris", 2020: "COVID-19"}
for yr, label in events.items():
    row = df_world[df_world["year"] == yr]
    if not row.empty:
        fig.add_annotation(x=yr, y=row["co2"].values[0], text=label,
                          showarrow=True, arrowhead=2, font=dict(size=9))
fig.show()
fig.write_image(ASSETS / "01_emissions_mondiales.png", scale=2)

### 4.2 Top émetteurs

In [ ]:
latest_year = df_countries["year"].max()
top10 = df_countries[df_countries["year"] == latest_year].nlargest(10, "co2")["country"].tolist()
print(f"Top 10 en {latest_year} : {top10}")

top5 = top10[:5]
df_top5 = df_countries[df_countries["country"].isin(top5)]

fig = px.line(
    df_top5, x="year", y="co2", color="country",
    title=f"Top 5 émetteurs — Évolution",
    labels={"year": "Année", "co2": "CO₂ (Mt)", "country": "Pays"},
    template="plotly_white", height=500,
)
fig.show()
fig.write_image(ASSETS / "02_top5_evolution.png", scale=2)

### 4.3 Émissions par habitant

In [ ]:
df_latest = df_countries[
    (df_countries["year"] == latest_year) & (df_countries["population"] > 1_000_000)
].copy()

top15_pc = df_latest.nlargest(15, "co2_per_capita")

fig = px.bar(
    top15_pc.sort_values("co2_per_capita"), x="co2_per_capita", y="country",
    orientation="h",
    title=f"Top 15 — CO₂ par habitant ({latest_year}, pays > 1M hab.)",
    labels={"co2_per_capita": "CO₂/hab (t)", "country": ""},
    color="co2_per_capita", color_continuous_scale="OrRd",
    template="plotly_white",
)
fig.update_layout(height=500, showlegend=False)
fig.show()
fig.write_image(ASSETS / "03_co2_par_habitant.png", scale=2)

### 4.4 Corrélation PIB × Émissions

In [ ]:
df_scatter = df_latest.dropna(subset=["gdp", "co2_per_capita", "population"])

fig = px.scatter(
    df_scatter, x="gdp", y="co2_per_capita", size="population",
    color="co2_per_capita", hover_name="country", log_x=True,
    title=f"PIB vs. CO₂ par habitant ({latest_year})",
    labels={"gdp": "PIB ($, log)", "co2_per_capita": "CO₂/hab (t)"},
    color_continuous_scale="OrRd", template="plotly_white", height=550,
)
fig.show()
fig.write_image(ASSETS / "04_pib_vs_co2.png", scale=2)

### 4.5 Découplage PIB / CO₂

In [ ]:
decouple = ["France", "Germany", "United Kingdom", "United States", "Japan"]
df_dec = df_countries[df_countries["country"].isin(decouple) & (df_countries["year"] >= 1990)].copy()

for c in decouple:
    mask = df_dec["country"] == c
    base_co2 = df_dec.loc[mask & (df_dec["year"] == 1990), "co2"].values
    base_gdp = df_dec.loc[mask & (df_dec["year"] == 1990), "gdp"].values
    if len(base_co2) > 0 and base_co2[0] > 0:
        df_dec.loc[mask, "co2_idx"] = df_dec.loc[mask, "co2"] / base_co2[0] * 100
    if len(base_gdp) > 0 and base_gdp[0] > 0:
        df_dec.loc[mask, "gdp_idx"] = df_dec.loc[mask, "gdp"] / base_gdp[0] * 100

df_fr = df_dec[df_dec["country"] == "France"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_fr["year"], y=df_fr["gdp_idx"],
    name="PIB (base 100)", line=dict(color="#3B7DD8")))
fig.add_trace(go.Scatter(x=df_fr["year"], y=df_fr["co2_idx"],
    name="CO₂ (base 100)", line=dict(color="#D64045")))
fig.add_hline(y=100, line_dash="dash", line_color="gray")
fig.update_layout(title="Découplage PIB / CO₂ — France (base 100 = 1990)",
    xaxis_title="Année", yaxis_title="Indice", template="plotly_white", height=450)
fig.show()
fig.write_image(ASSETS / "05_decouplage_france.png", scale=2)

### 4.6 Émissions par source d'énergie

In [ ]:
energy_cols = ["coal_co2", "oil_co2", "gas_co2", "cement_co2", "flaring_co2"]
existing_e = [c for c in energy_cols if c in df_world.columns]
df_energy = df_world[["year"] + existing_e].dropna(subset=existing_e)

colors = {"coal_co2": "#2D3436", "oil_co2": "#D64045", "gas_co2": "#3B7DD8",
          "cement_co2": "#E8913A", "flaring_co2": "#95A5A6"}
labels_e = {"coal_co2": "Charbon", "oil_co2": "Pétrole", "gas_co2": "Gaz",
            "cement_co2": "Ciment", "flaring_co2": "Torchage"}

fig = go.Figure()
for col in existing_e:
    fig.add_trace(go.Scatter(x=df_energy["year"], y=df_energy[col],
        name=labels_e.get(col, col), stackgroup="one",
        line=dict(color=colors.get(col))))
fig.update_layout(title="Émissions mondiales par source",
    xaxis_title="Année", yaxis_title="CO₂ (Mt)",
    template="plotly_white", height=500)
fig.show()
fig.write_image(ASSETS / "06_emissions_par_source.png", scale=2)

### 4.7 Trajectoires par continent

In [ ]:
continents = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]
df_cont = df_regions[df_regions["country"].isin(continents)]

fig = px.area(df_cont, x="year", y="co2", color="country",
    title="Émissions CO₂ par continent",
    labels={"year": "Année", "co2": "CO₂ (Mt)", "country": "Continent"},
    template="plotly_white", height=500)
fig.show()
fig.write_image(ASSETS / "07_emissions_continents.png", scale=2)

### 4.8 Intensité carbone (CO₂/PIB)

In [ ]:
i_countries = ["China", "India", "United States", "France", "Germany", "Brazil"]
df_int = df_countries[
    df_countries["country"].isin(i_countries) &
    df_countries["co2_per_gdp"].notna() & (df_countries["year"] >= 1990)
]
fig = px.line(df_int, x="year", y="co2_per_gdp", color="country",
    title="Intensité carbone (1990–2023)",
    labels={"year": "Année", "co2_per_gdp": "CO₂/PIB (kg/$)", "country": "Pays"},
    template="plotly_white", height=500)
fig.show()
fig.write_image(ASSETS / "08_intensite_carbone.png", scale=2)

## 5. Export pour Looker Studio

In [ ]:
looker_cols = {
    "country": "Pays", "iso_code": "Code_ISO", "year": "Annee",
    "population": "Population", "gdp": "PIB",
    "co2": "CO2_Total_Mt", "co2_per_capita": "CO2_Par_Habitant",
    "co2_per_gdp": "Intensite_Carbone", "share_global_co2": "Part_Mondiale_Pct",
    "coal_co2": "CO2_Charbon", "oil_co2": "CO2_Petrole",
    "gas_co2": "CO2_Gaz", "cement_co2": "CO2_Ciment",
    "energy_per_capita": "Energie_Par_Habitant",
}
existing_lk = {k: v for k, v in looker_cols.items() if k in df_countries.columns}
df_export = df_countries[list(existing_lk.keys())].rename(columns=existing_lk)
df_export = df_export.dropna(subset=["CO2_Total_Mt"])

for col in df_export.select_dtypes(include=["float64"]).columns:
    df_export[col] = df_export[col].round(3)

nb_cells = len(df_export) * df_export.shape[1]
print(f"Export : {len(df_export):,} lignes × {df_export.shape[1]} colonnes = {nb_cells:,} cellules")
if nb_cells > 5_000_000:
    print("⚠️  Trop pour Google Sheets — filtrage 30 dernières années")
    df_export = df_export[df_export["Annee"] >= df_export["Annee"].max() - 30]

output = DATA_PROCESSED / "emissions_looker.csv"
df_export.to_csv(output, index=False, encoding="utf-8-sig")
print(f"\n✓ Exporté : {output} ({output.stat().st_size / 1e6:.1f} Mo)")
display(df_export.head())

---

*Notebook réalisé par Diogo Almeida — Septembre 2026*
*Données : Our World in Data — Creative Commons BY*